# 05 · The graph the model writes

Same fan-out as chapter 04. One thing moves: a planning call reads the incident first
and writes the prompt each child will be given.

> **You'll learn**
> - Meta-prompt a graph: a reasoner whose output is other reasoners' input
> - Pass runtime-authored prompts as kwargs into one generic worker
> - Compare the questions two different incidents provoke

In [1]:
import sys, json, requests
sys.path.insert(0, "../lib")
import dag

SERVER = "http://localhost:8080"

def run(reasoner, incident_id):
    """Drive the node through the control plane. Never `await app.call` from a notebook."""
    r = requests.post(f"{SERVER}/api/v1/execute/blast-radius.{reasoner}",
                      json={"input": {"incident_id": incident_id}}, timeout=600)
    r.raise_for_status()
    d = r.json()
    print(f"{incident_id}  {d['status']}  {d['duration_ms']/1000:.0f}s  run={d['run_id']}")
    return d

dag.print_nodes()

   (10 unrelated agent(s) on this control plane, not shown)
   1 agent(s) registered:
     - blast-radius [active] 24 reasoner(s) @ http://127.0.0.1:8002
   NOTE: registrations outlive the process. health_status can lie;
         dag.node_alive(<agent_id>) pings the node itself.


`r05_plan` returns a list of `{lens_name, prompt, why}`. `diagnose` then calls the
single generic `r05_lens` once per spec, capped at six.

In [2]:
src = open("../node/rungs/r05.py").read()
print(src[src.index('@router.reasoner(tags=["entry"])'):])

@router.reasoner(tags=["entry"])
async def diagnose(incident_id: str, model: str | None = None) -> Diagnosis:
    """Plan the graph, run it, merge it."""
    graph = await router.app.call(f"{NODE_ID}.r05_plan", incident_id=incident_id, model=model)
    specs = (graph.get("lenses") or [])[:MAX_LENSES]
    reports = await asyncio.gather(
        *(
            router.app.call(
                f"{NODE_ID}.r05_lens",
                incident_id=incident_id,
                lens_name=s["lens_name"],
                prompt=s["prompt"],
                model=model,
            )
            for s in specs
        )
    )
    return await router.app.call(
        f"{NODE_ID}.r05_synthesize", incident_id=incident_id, reports=list(reports), model=model
    )



Run the same two incidents chapter 04 ran.

In [3]:
a = run("r05_diagnose", "inc-008")
b = run("r05_diagnose", "inc-011")

inc-008  succeeded  54s  run=run_20260820_123230_cvu3fldb


inc-011  succeeded  41s  run=run_20260820_123324_8qtbs0pf


## The generated prompts

This is the output that matters. Nobody wrote these questions; the model did, after
reading the incident.

In [4]:
import ast, textwrap

def plan_of(d):
    """The plan is the first child execution of the run."""
    ex = dag.fetch_run(d["run_id"])["executions"]
    for e in ex:
        if e.get("reasoner_id", "").endswith("r05_plan"):
            out = e.get("result") or {}
            if isinstance(out, str):
                try:
                    out = json.loads(out)
                except json.JSONDecodeError:
                    out = ast.literal_eval(out)
            return out
    return {}

def show(label, d):
    p = plan_of(d)
    print("=" * 78)
    print(f"{label}   symptom shape: {p.get('symptom_shape','?')}")
    print("=" * 78)
    for s in p.get("lenses", []):
        print(f"\n[{s['lens_name']}]  {s.get('why','')}")
        print(textwrap.fill(s["prompt"], 76, initial_indent="    ", subsequent_indent="    "))
    print()

show("inc-008 · memory leak", a)
show("inc-011 · auth failures", b)

inc-008 · memory leak   symptom shape: Memory high, CPU flat, restarts up, consumer lag up

[change_correlation]  The heap snapshot shows 1.84M EventEmitter[render] listeners, and logs show MaxListenersExceededWarning; the only recent deploy that touched render and metrics is dep-2201.
    Examine the code change in dep-2201 (version 5.2.0 of notification-
    worker) that added per-recipient template personalisation and a render
    metrics hook. Verify if the new code in TemplateRenderer.ts registers a
    listener on the shared metrics emitter for every render call but never
    removes it, leading to accumulating event listeners. Check the git diff
    for the metrics hook registration and any cleanup logic.

[memory_lifecycle]  The OOM is caused by a memory leak; the heap snapshot pinpoints the top retainer as EventEmitter listeners.
    Analyze the heap snapshot taken at 07:38 UTC showing 1.84 million
    EventEmitter[render] listeners retaining 812 MB. Investigate the code in
  

Different lenses, and — more to the point — different *questions*. The prompts name
this incident's pods, series and timestamps. A prompt that would read the same for any
incident is a wasted node.

In [5]:
def lenses(d):
    return [s["lens_name"] for s in plan_of(d).get("lenses", [])]

la, lb = lenses(a), lenses(b)
print("inc-008:", la)
print("inc-011:", lb)
print("shared :", sorted(set(la) & set(lb)))
print("only 008:", sorted(set(la) - set(lb)))
print("only 011:", sorted(set(lb) - set(la)))

inc-008: ['change_correlation', 'memory_lifecycle', 'long_horizon_change', 'timeline', 'alert_validity']
inc-011: ['clock_time', 'long_horizon_change', 'change_correlation', 'blast_scope', 'resource_contention', 'observability_gap']
shared : ['change_correlation', 'long_horizon_change']
only 008: ['alert_validity', 'memory_lifecycle', 'timeline']
only 011: ['blast_scope', 'clock_time', 'observability_gap', 'resource_contention']


## The two graphs

In [6]:
dag.render_two(a["run_id"], b["run_id"], labels=("inc-008 · memory leak", "inc-011 · auth failures"))

```mermaid
flowchart LR
  subgraph ag["inc-008 · memory leak — 8 exec · depth 2 · fan-out 7"]
  direction TD
    a0["r05_diagnose<br/><small>✓ succeeded · 54.0s</small>"]
    a1["r05_plan<br/><small>✓ succeeded · 17.8s</small>"]
    a2["r05_lens<br/><small>✓ succeeded · 4.2s</small>"]
    a3["r05_lens<br/><small>✓ succeeded · 6.0s</small>"]
    a4["r05_lens<br/><small>✓ succeeded · 13.4s</small>"]
    a5["r05_lens<br/><small>✓ succeeded · 18.3s</small>"]
    a6["r05_lens<br/><small>✓ succeeded · 5.6s</small>"]
    a7["r05_synthesize<br/><small>✓ succeeded · 17.0s</small>"]
    a0 --> a1
    a0 --> a2
    a0 --> a3
    a0 --> a4
    a0 --> a5
    a0 --> a6
    a0 --> a7
    class a0,a1,a2,a3,a4,a5,a6,a7 ok;
  end
  subgraph bg["inc-011 · auth failures — 9 exec · depth 2 · fan-out 8"]
  direction TD
    b0["r05_diagnose<br/><small>✓ succeeded · 41.0s</small>"]
    b1["r05_plan<br/><small>✓ succeeded · 13.5s</small>"]
    b2["r05_lens<br/><small>✓ succeeded · 6.0s</small>"]
    b3["r05_lens<br/><small>✓ succeeded · 10.2s</small>"]
    b4["r05_lens<br/><small>✓ succeeded · 11.9s</small>"]
    b5["r05_lens<br/><small>✓ succeeded · 9.1s</small>"]
    b6["r05_lens<br/><small>✓ succeeded · 8.8s</small>"]
    b7["r05_lens<br/><small>✓ succeeded · 4.9s</small>"]
    b8["r05_synthesize<br/><small>✓ succeeded · 14.4s</small>"]
    b0 --> b1
    b0 --> b2
    b0 --> b3
    b0 --> b4
    b0 --> b5
    b0 --> b6
    b0 --> b7
    b0 --> b8
    class b0,b1,b2,b3,b4,b5,b6,b7,b8 ok;
  end
  classDef ok   fill:#dcfce7,stroke:#16a34a,stroke-width:1px,color:#14532d;
  classDef run  fill:#dbeafe,stroke:#2563eb,stroke-width:1px,color:#1e3a8a;
  classDef wait fill:#f1f5f9,stroke:#94a3b8,stroke-width:1px,color:#334155;
  classDef bad  fill:#fee2e2,stroke:#dc2626,stroke-width:1px,color:#7f1d1d;
```

Both plans came out six lenses wide, so the pictures still rhyme. What changed is
underneath: half the nodes are different lenses, and every node was handed a different
question. Chapter 04's two runs were the same picture *and* the same questions.


In [7]:
for label, d in (("inc-008", a), ("inc-011", b)):
    r = d["result"]
    print(f"--- {label}  (confident={r['confident']})")
    print("root cause:", r["root_cause"])
    print()

--- inc-008  (confident=True)
root cause: The dep-2201 deploy added a metrics hook in TemplateRenderer.ts:214 that registers a listener on the shared module-level EventEmitter per render call without ever removing it, causing 1.84 million accumulated listeners to retain 812 MB of memory, eventually exhausting heap and triggering OOM kills.

--- inc-011  (confident=True)
root cause: Clock drift on node eu-c1-n07 due to chronyd losing NTP synchronization during a live migration, causing JWT validation and webhook signature failures from time skew exceeding the 30s leeway.



## What you learned

- **A planning reasoner can author its children's prompts** and pass them as ordinary kwargs.
- **One generic worker plus runtime prompts** gives you a variable graph without writing a node per lens.
- **The questions now depend on the input** — which is exactly the process variance chapter 08 measures.

**Next:** 06 · let the model grow the graph as findings arrive, instead of planning it all up front.